In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS
from config import IST


def get_video_datetime(video_path: Path) -> tuple[datetime | None, str]:
    """Extract the datetime when the video was recorded from metadata.

    Args:
        video_path: Path to the video file

    Returns:
        tuple: (datetime object or None, formatted string for display)
    """
    if not video_path.exists() or not video_path.is_file():
        print(f"File not found: {video_path}")
        return None, "Unknown"

    try:
        command = [
            "ffprobe",
            "-v",
            "quiet",
            "-print_format",
            "json",
            "-show_format",
            "-show_streams",
            str(video_path),
        ]
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
        )
        metadata = json.loads(result.stdout.decode())

        # Try to get creation time from format tags
        if "format" in metadata and "tags" in metadata["format"]:
            tags = metadata["format"]["tags"]
            # Common metadata fields for creation time
            for key in ["com.apple.quicktime.creationdate", "creation_time", "date"]:
                if key in tags:
                    try:
                        # Parse ISO 8601 format (most common)
                        dt_str = tags[key]
                        is_utc = dt_str.endswith("Z")
                        # Remove 'Z' timezone indicator for parsing
                        dt_str = dt_str.replace("Z", "").strip()
                        # Handle different datetime formats
                        for fmt in [
                            "%Y-%m-%dT%H:%M:%S.%f%z",
                            "%Y-%m-%dT%H:%M:%S%z",
                            "%Y-%m-%dT%H:%M:%S.%f",
                            "%Y-%m-%dT%H:%M:%S",
                            "%Y-%m-%d %H:%M:%S",
                        ]:
                            try:
                                dt = datetime.strptime(dt_str, fmt)
                                # If the original string had 'Z', it's UTC time - convert to IST
                                if is_utc:
                                    utc_tz = timezone.utc
                                    dt_utc = dt.replace(tzinfo=utc_tz)
                                    dt = dt_utc.astimezone(IST)
                                # If datetime already has timezone info, convert to IST
                                elif dt.tzinfo is not None:
                                    dt = dt.astimezone(IST)
                                formatted = dt.strftime("%B %d, %Y at %I:%M:%S %p")
                                return dt, formatted
                            except ValueError:
                                continue
                    except Exception:
                        continue
    except Exception:
        pass
    return None, "Unknown"


In [ ]:
get_video_datetime(Path("/Users/bisane.s/my_files/my_codes/github/image-conversion/files/to_convert/IMG_0489.MOV"))


(datetime.datetime(2025, 12, 15, 20, 30, 27, tzinfo=datetime.timezone(datetime.timedelta(seconds=19800))),
 'December 15, 2025 at 08:30:27 PM')

In [ ]:
def get_audio_bitrate(source_path: Path) -> str:
    """Get the audio bitrate of a video file.

    Args:
        source_path: Path to the video file

    Returns:
        Audio bitrate as a string (e.g., "128000"), or "0" if unable to determine
    """
    try:
        command = [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "a:0",
            "-show_entries",
            "stream=bit_rate",
            "-of",
            "default=noprint_wrappers=1:nokey=1",
            str(source_path),
        ]
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
        )
        bitrate = result.stdout.decode().strip()
        return bitrate if bitrate else "0"
    except (subprocess.CalledProcessError, ValueError):
        return "0"


In [ ]:
get_audio_bitrate(Path("/Users/bisane.s/my_files/my_codes/github/image-conversion/files/to_convert/IMG_0489.MOV"))


'183937'

In [ ]:
def get_video_quality(source_path: Path) -> str:
    """Get the video quality (resolution) of a video file.

    Args:
        source_path: Path to the video file

    Returns:
        Video quality as a string (e.g., "1920x1080"), or "Unknown" if unable to determine
    """
    try:
        command = [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "v:0",
            "-show_entries",
            "stream=width,height",
            "-of",
            "csv=p=0",
            str(source_path),
        ]
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
        )
        quality = result.stdout.decode().strip()
        return quality if quality else "Unknown"
    except (subprocess.CalledProcessError, ValueError):
        return "Unknown"


In [ ]:
get_video_quality(Path("/Users/bisane.s/my_files/my_codes/github/image-conversion/files/to_convert/IMG_0489.MOV"))


'Unknown'